# Computing expected values

In this notebook we use Python to compute expected values of random variables.
This is a very important "ingredient" in dynamic programming methods,
used to evaluate and find policies in small Markov Decision Process settings.

*<span style="color:red">Below, the parts indicated by `#??` need to be filled in!</span>*

### Example 1: normal dice

Consider a fair, 6-sided dice. We want to find the expected value of the number shown after a roll.
This can easily be computed to be $\sum_{k=1}^6 1/6 \cdot k = 3.5$.
In more complicated settings, it is often easier to iterate over possible outcomes and compute the expectation in a more naive fashion.

In [1]:
## Function that computes the expected payout
def ePayout():
    numbers = [1, 2, 3, 4, 5, 6] # ?? Possible numbers shown
    probs = [1/6, 1/6, 1/6, 1/6, 1/6, 1/6] # ?? Probability of each number
    E = 0
    for n, p in zip(numbers, probs):
        E += n * p # ?? Compute "contribution" of this outcome
    return E

print(ePayout())

3.5


### Example 2: $n$-sided dice

Consider the following "game":
The player rolls a fair $n$-sided dice (labelled $1, 2, \dots, n$) and receives a payout of $X^2$,
where $X$ denotes the number shown by the dice.
What is the expected payout?

In [10]:
## Function that computes the expected payout for n
import numpy as np
def ePayout(n):
    numbers = np.arange(1, n+1)
    numbers = np.power(numbers, 2)
    probs = np.ones(n) / n 
    E = 0
    for i, p in zip(numbers, probs):
        E += i * p 
    return E

# Test for some value of n
print(ePayout(7))

20.0


*Bonus: this value can also be derived algebraically, using the identity $\sum_{x=1}^n x^2 = n(n+1)(2n+1)/6$.*

### Example 3: two-step game

Consider the following "game":
The player rolls a fair $n$-sided dice,
let $X$ denote the result of this roll.
If $X$ is odd, the player receives a payout of $-1$.
If $X$ is even, the player draws a card from a deck labelled $(-2, -1, 0, 1, \dots, X/2)$
and receives a payout equal to the number drawn.
What is the expected payout?

In [28]:
## Function that computes the expected payout for n, m
def ePayout(n):
    probs = np.ones(n) / n
    payout = np.empty(n)
    for i in range(1, n+1):
        if i % 2 != 0:
            payout[i-1] = -1
        else:
            payout[i-1] = np.sum(np.arange(-2, i/2 + 1)) / np.size(np.arange(-2, i/2 + 1))
    E = 0
    for i, p in zip(payout, probs):
        E += i * p
    print(payout)
    return E

# Check values for some n
for x in range(1, 5):
    print(f'{ePayout(x):.4f}')

[-1.]
-1.0000
[-1.  -0.5]
-0.7500
[-1.  -0.5 -1. ]
-0.8333
[-1.  -0.5 -1.   0. ]
-0.6250


### Monte Carlo

The expectations above can also be estimated using Monte-Carlo estimation.
To this end,
simulate the experiment $N$ times (for some large $N$),
record the outcomes and compute the average.
This average will converge to the true expectation as $N$ grows.

In [11]:
# We need to generate random numbers to do this
import random

In [34]:
## Example 1
def mcPayout_1(N):
    #?? Create an empty list for the results
    results = []
    #?? Loop N times
    for _ in range(N):
        num = np.random.randint(1, 7)  # Generate a random number between 1 and 6
        #?? Append the number to the list
        results.append(num)
    #?? Compute the average of the list
    return np.mean(results)

# Run MC
mcPayout_1(10000)

3.4996

In [36]:
## Example 2
def mcPayout_2(N, n):
    results = []
    for _ in range(N):
        num = np.random.randint(1, n+1)  # Generate a random number between 1 and n
        results.append(num**2)  # Append the square of the number to the list
    return np.mean(results)  # Compute and return the average of the list
mcPayout_2(1000, 7)

19.917

In [39]:
## Example 3
def mcPayout_3(N, n):
    results = []
    for _ in range(N):
        num = np.random.randint(1, n+1)  # Generate a random number between 1 and n
        if num % 2 != 0:
            results.append(-1)
        else:
            results.append(np.random.uniform(-2, num/2))  # Append a random number between -2 and num/2 to the list
    return np.mean(results)  # Compute and return the average of the list

# Run MC
for x in range(1, 5):
    print(mcPayout_3(1000, x).round(4))

-1.0
-0.7721
-0.8195
-0.6365
